# Bielik - podstawy: chat, structured outputs, RAG po polsku

[Bielik](https://bielik.ai/) to polski model językowy stworzony przez [SpeakLeash](https://speakleash.org/) i ICM. W tym notebooku poznamy go w trzech praktycznych zastosowaniach - od najprostszego chatu, przez wymuszanie ustrukturyzowanego JSON-a, aż po RAG po polsku z polskimi embeddingami.

Notebook świadomie nie korzysta z żadnego frameworka agentowego - używamy tych samych prymitywów, które studenci znają z `001-01. LLM-OpenAI.ipynb`, tylko skierowanych na lokalnie uruchomionego Bielika przez [Ollamę](https://ollama.com/). Notebooki `010-01..04` pokazują, jak ten sam Bielik gra w 4 różnych frameworkach agentowych.

## Plan notebooka

1. **Sekcja A - prosty chat**: pierwsze wywołanie modelu, system prompt po polsku, wpływ parametru `temperature`.
2. **Sekcja B - structured outputs**: ekstrakcja danych z tekstu encyklopedycznego do modelu Pydantica, z walidacją.
3. **Sekcja C - RAG po polsku**: lokalny pipeline retrievalu na 50 notatkach o miastach, z polskim embedderem `mmlw` i Bielikiem jako generatorem odpowiedzi.

> **Uwaga dot. DevContainera:** Notebook działa wewnątrz kontenera Dockera, więc `localhost` z perspektywy notebooka wskazuje na kontener, a nie hosta z uruchomioną Ollamą. W kodzie używamy `host.docker.internal`, aby z wnętrza kontenera dotrzeć do Ollamy działającej na hoście.

## Wymagania

1. Zainstalowana Ollama: https://ollama.com/download
2. Pobrany model Bielika: `ollama pull SpeakLeash/bielik-11b-v3.0-instruct:bf16`
3. Zbudowany model `bielik-tools` z customowego Modelfile (`010-00. Bielik.Modelfile`):
   ```bash
   ollama create bielik-tools -f "010-00. Bielik.Modelfile"
   ```
   Modelfile naprawia bug ze stop tokens (oryginał używa tokenów Llamy 3 zamiast ChatML, co psuje generację). Szczegóły w `010-00. Bielik.Modelfile.md`.
4. Działający serwis Ollama w tle (port `11434`).
5. Pakiety Pythona: `openai`, `pydantic`, `sentence-transformers`, `numpy` (w środowisku Dockera kursu są już zainstalowane).

> **Uwaga dot. embeddera:** Pierwsze użycie modelu `sdadas/mmlw-retrieval-roberta-large` w sekcji C pobierze ~1,4 GB wag z HuggingFace. Kolejne uruchomienia korzystają z cache.

## Konfiguracja klienta OpenAI

Używamy tego samego klienta `OpenAI`, którego znamy z `001-01. LLM-OpenAI.ipynb` - tyle, że wskazanego na lokalną Ollamę przez OpenAI-kompatybilny endpoint. To ten sam wzorzec, którego używają wszystkie notebooki sekcji 010.

In [ ]:
import os
import json
from openai import OpenAI

# Klient OpenAI wskazujący na lokalną Ollamę (kompatybilny endpoint /v1).
client = OpenAI(
    base_url="http://host.docker.internal:11434/v1",
    api_key="ollama",  # atrapa - Ollama ignoruje, ale klient OpenAI wymaga niepustego pola
)

# Nazwa modelu zbudowanego z customowego Modelfile.
MODEL = "bielik-tools"

## A. Pierwsze wywołanie - prosty chat

Najprostsze możliwe użycie - jeden system prompt, jedno pytanie, jedna odpowiedź. Bielik świetnie rozumie polski (bo był na nim trenowany), więc całą rozmowę prowadzimy po polsku.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Jesteś rzeczowym asystentem. Odpowiadasz krótko i konkretnie po polsku."},
        {"role": "user", "content": "Wyjaśnij w 2 zdaniach, czym jest model Bielik."},
    ],
    temperature=0.3,
)

print(response.choices[0].message.content)

### Wpływ parametru `temperature`

Parametr `temperature` kontroluje losowość generowanego tekstu - im wyższy, tym bardziej kreatywne i nieprzewidywalne odpowiedzi. Pokażmy to na krótkim wierszyku o Poznaniu: raz z `temperature=0.0` (deterministycznie), raz z `temperature=0.9` (kreatywnie).

In [ ]:
def generate_poem(temperature: float) -> str:
    """Krótki wierszyk o Poznaniu z zadaną temperaturą."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Jesteś poetą. Piszesz krótkie, czterowersowe rymowanki po polsku."},
            {"role": "user", "content": "Napisz krótki wierszyk o Poznaniu."},
        ],
        temperature=temperature,
    )
    return response.choices[0].message.content

print("=== temperature = 0.0 ===")
print(generate_poem(0.0))

print("\n=== temperature = 0.9 ===")
print(generate_poem(0.9))

## B. Structured outputs - ekstrakcja z tekstu encyklopedycznego

Bielik na Ollamie wspiera wymuszanie struktury odpowiedzi przez parametr `response_format` z JSON Schema - dokładnie tak jak GPT-4o w `004-01. LLM-structured_outputs.ipynb`. Wykorzystamy to do **ekstrakcji ustrukturyzowanych danych** z naturalnego tekstu encyklopedycznego po polsku.

Przepis jest standardowy:
1. Definiujemy schemat danych jako klasę Pydantica.
2. Wstrzykujemy `model_json_schema()` w `response_format`.
3. Walidujemy odpowiedź modelu przez `model_validate_json()`.

### Schemat danych (Pydantic)

Definiujemy schemat informacji o kraju - pola po angielsku (zgodnie z konwencją API), opisy po polsku (żeby model rozumiał, co ma wypełnić). Dzięki `Field(description=...)` opisy trafiają do JSON Schema i służą modelowi jako podpowiedzi.

In [ ]:
from pydantic import BaseModel, Field


class CountryInfo(BaseModel):
    """Wyciąg najważniejszych faktów o państwie z tekstu encyklopedycznego."""

    name: str = Field(description="Oficjalna nazwa kraju w języku polskim")
    capital: str = Field(description="Stolica kraju")
    population_mln: float | None = Field(description="Populacja w milionach mieszkańców")
    area_km2: int | None = Field(description="Powierzchnia kraju w kilometrach kwadratowych")
    official_languages: list[str] = Field(description="Lista języków urzędowych")
    currency: str | None = Field(description="Oficjalna waluta kraju")

### Tekst wejściowy

Krótki paragraf encyklopedyczny po polsku - z fragmentami liczbowymi, listami i opisem. Naszym zadaniem jest wyciągnąć z niego wartości pól zdefiniowanego wyżej schematu.

In [ ]:
text = """
Estonia, oficjalnie Republika Estońska, to państwo w północno-wschodniej Europie
leżące nad Morzem Bałtyckim. Stolicą i największym miastem kraju jest Tallinn, którego
Stare Miasto wpisano na listę UNESCO. Estonia liczy około 1,37 miliona mieszkańców
i zajmuje powierzchnię 45 339 km². Jedynym językiem urzędowym jest estoński, choć
duża część populacji posługuje się także rosyjskim. Walutą kraju od 2011 roku jest euro,
po wejściu Estonii do strefy euro. Estonia jest członkiem Unii Europejskiej oraz NATO.
"""

print(text)

### Wywołanie modelu z `response_format`

Kluczowy moment: do `response_format` przekazujemy **bezpośrednio klasę Pydantica** - SDK OpenAI sam wygeneruje z niej JSON Schema, wyśle go w żądaniu i sparsuje odpowiedź do obiektu Pythona. To ten sam wzorzec, który w `004-01. LLM-structured_outputs.ipynb` był pokazany przez `client.responses.parse(text_format=...)`; tutaj używamy odpowiednika z Chat Completions API, bo Ollama wspiera tylko ten endpoint.

In [ ]:
response = client.chat.completions.parse(  # parse zamiast create - SDK sam zbuduje schemę i sparsuje odpowiedź
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "Jesteś asystentem ekstraktującym ustrukturyzowane dane z tekstu encyklopedycznego. "
                "Zwracasz wyłącznie poprawny JSON zgodny ze schematem - bez dodatkowych komentarzy."
            ),
        },
        {
            "role": "user",
            "content": f"Wyciągnij informacje o kraju z poniższego tekstu:\n\n{text}",
        },
    ],
    response_format=CountryInfo,  # przekazujemy klasę Pydantic - SDK sam generuje schemę i parsuje odpowiedź
    temperature=0.0,
)

### Wynik - od razu obiekt Pythona

Dzięki `parse(...)` w odpowiedzi mamy gotowy obiekt `CountryInfo` pod `response.choices[0].message.parsed`. Nie musimy ręcznie wołać `model_validate_json` - SDK robi to za nas.

In [ ]:
country = response.choices[0].message.parsed

print(country.model_dump_json(indent=2, ensure_ascii=False))

# Możemy teraz korzystać z atrybutów jak ze zwykłego obiektu Pythona:
print(f"\nStolicą kraju {country.name} jest {country.capital}.")
print(f"Powierzchnia: {country.area_km2} km², populacja: {country.population_mln} mln.")

## C. RAG po polsku - z polskimi embeddingami i Bielikiem

Sekcja 008 (`008-01..03`) pokazuje RAG z OpenAI i Weaviate. Tutaj zrobimy to samo **w pełni lokalnie i po polsku** - z polskim embedderem `sdadas/mmlw-retrieval-roberta-large` i Bielikiem jako generatorem odpowiedzi.

Korzystamy z małej bazy 50 notatek o miastach (`010-00. Bielik-podstawy.json`), więc nie potrzebujemy Weaviate ani FAISS - cosine similarity policzymy w `numpy`.

### Architektura

![Architektura RAG po polsku](assets/010-00.%20Architektura%20RAG.png)

### Załadowanie korpusu

Wczytujemy 50 obiektów `{"city": ..., "note": ...}` z pliku JSON obok notebooka.

In [ ]:
with open("010-00. Bielik-podstawy.json") as f:
    corpus = json.load(f)

print(f"Wczytano {len(corpus)} notatek")
print("\nPrzykładowe rekordy:")
for item in corpus[:3]:
    print(f"  - {item['city']}: {item['note'][:80]}...")

### Embedder polski - `mmlw`

Używamy modelu [`sdadas/mmlw-retrieval-roberta-large`](https://huggingface.co/sdadas/mmlw-retrieval-roberta-large) - polskiego embeddera retrievalowego od Sławomira Dadasa, świetnego w polskich benchmarkach.

Specyfika tego modelu: tak jak inne modele z rodziny `e5` / `mmlw`, **wymaga prefixów** rozróżniających role tekstu:
- `query:` - dla pytania użytkownika,
- `passage:` - dla dokumentów w korpusie.

Bez tych prefixów retrieval działa znacząco gorzej.

In [ ]:
from sentence_transformers import SentenceTransformer

# Pierwsze użycie pobierze ~1.4 GB wag z HuggingFace.
embedder = SentenceTransformer("sdadas/mmlw-retrieval-roberta-large")

print(f"Wymiarowość embeddingów: {embedder.get_sentence_embedding_dimension()}")

### Embedowanie korpusu

Każdą notatkę poprzedzamy prefixem `passage:` i embedujemy w jednej partii. `normalize_embeddings=True` daje wektory długości 1, więc cosine similarity sprowadza się do zwykłego iloczynu skalarnego.

In [ ]:
import numpy as np

passage_texts = [f"passage: {item['note']}" for item in corpus]
passage_embs = embedder.encode(passage_texts, normalize_embeddings=True)

print(f"Kształt macierzy embeddingów: {passage_embs.shape}")

### Funkcja `retrieve`

Liczy cosine similarity między pytaniem (z prefixem `query:`) a wszystkimi notatkami i zwraca `k` najbardziej podobnych. Ponieważ embeddingi są znormalizowane, iloczyn macierz × wektor daje od razu podobieństwa.

In [ ]:
def retrieve(query: str, k: int = 3) -> list[dict]:
    """Zwraca top-k notatek z korpusu najbardziej pasujących do pytania."""
    query_emb = embedder.encode(f"query: {query}", normalize_embeddings=True)
    sims = passage_embs @ query_emb
    top_idx = sims.argsort()[-k:][::-1]
    return [corpus[i] for i in top_idx]


# Test retrievalu - same notatki bez wywoływania Bielika.
for item in retrieve("Gdzie znajduje się Wawel?", k=3):
    print(f"  - {item['city']}: {item['note']}")

### Pętla RAG

Łączymy retrieval z generacją: pobieramy top-k notatek, wstrzykujemy je do system promptu jako kontekst i prosimy Bielika o odpowiedź. Model ma wyraźną instrukcję, żeby korzystać **wyłącznie** z dostarczonych notatek - to klasyczne ograniczenie RAG-a, które redukuje halucynacje.

In [ ]:
def rag(question: str, k: int = 3) -> str:
    """RAG: retrieve top-k notatek + odpowiedź Bielika z kontekstem."""
    notes = retrieve(question, k=k)
    context = "\n".join(f"- {n['city']}: {n['note']}" for n in notes)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Odpowiadasz na pytania użytkownika korzystając WYŁĄCZNIE z poniższych notatek "
                    "o miastach. Jeśli odpowiedzi nie ma w notatkach, powiedz że nie wiesz.\n\n"
                    f"NOTATKI:\n{context}"
                ),
            },
            {"role": "user", "content": question},
        ],
        temperature=0.3,
    )
    return response.choices[0].message.content


# Trzy demo zapytania pokazujące działanie pełnego pipeline'u.
pytania = [
    "Gdzie znajduje się Wawel?",
    "Powiedz mi coś ciekawego o Toruniu.",
    "Które miasta z bazy leżą nad Wisłą?",
]

for q in pytania:
    print(f"=== Pytanie: {q} ===")
    print(rag(q))
    print()